# NVIDIA C-RADIOv4 for Segmentation

> Testing NVIDIA's C-RADIOv4 vision foundation model for segmentation tasks with custom data

**References:**
- [C-RADIOv4-H on HuggingFace](https://huggingface.co/nvidia/C-RADIOv4-H)
- [NVlabs/RADIO GitHub](https://github.com/NVlabs/RADIO)
- [C-RADIOv4 Technical Report](https://arxiv.org/abs/2601.17237)

## Overview

C-RADIOv4 (AM-RADIO: Agglomerative Model - Reduce All Domains Into One) is NVIDIA's vision foundation model that combines:
- **SigLIP2** - for text-image understanding
- **DINOv3** - for self-supervised visual features  
- **SAM3** - for segmentation capabilities

The model outputs:
- `summary`: Global image representation (B, C)
- `spatial_features`: Dense per-patch features (B, T, D) or (B, C, H, W) - ideal for segmentation

## 1. Setup and Installation

In [ ]:
# Install dependencies if needed
# !pip install torch torchvision transformers pillow matplotlib numpy scikit-learn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from torchvision.transforms.functional import pil_to_tensor
from typing import List, Tuple, Optional, Union
import warnings
warnings.filterwarnings('ignore')

# Check CUDA availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Load C-RADIOv4 Model

Two options available:
1. **TorchHub** (recommended for flexibility)
2. **HuggingFace** (easier setup)

### Option A: Load via TorchHub

In [ ]:
def load_radio_torchhub(version: str = "c-radio_v4-h"):
    """
    Load C-RADIOv4 model via TorchHub
    
    Available versions:
    - 'c-radio_v4-h': C-RADIOv4-H (653M params, ViT-H/16) - Best quality
    - 'c-radio_v4-so400m': C-RADIOv4-SO400M (431M params) - Good balance
    - 'c-radio_v3-h', 'c-radio_v3-l', 'c-radio_v3-b': Previous versions
    - 'radio_v2.5-g', 'radio_v2.5-h', 'radio_v2.5-l', 'radio_v2.5-b': v2.5 variants
    """
    print(f"Loading RADIO model: {version}")
    model = torch.hub.load(
        'NVlabs/RADIO', 
        'radio_model', 
        version=version,
        progress=True, 
        skip_validation=True
    )
    return model

# Load the model
try:
    model = load_radio_torchhub("c-radio_v4-h")
    model = model.to(device).eval()
    print("Model loaded successfully via TorchHub!")
    USE_TORCHHUB = True
except Exception as e:
    print(f"TorchHub loading failed: {e}")
    print("Will try HuggingFace instead...")
    USE_TORCHHUB = False

### Option B: Load via HuggingFace

In [ ]:
if not USE_TORCHHUB:
    from transformers import AutoModel, CLIPImageProcessor
    
    hf_repo = "nvidia/C-RADIOv4-H"
    print(f"Loading model from HuggingFace: {hf_repo}")
    
    model = AutoModel.from_pretrained(hf_repo, trust_remote_code=True)
    image_processor = CLIPImageProcessor.from_pretrained(hf_repo)
    model = model.to(device).eval()
    print("Model loaded successfully via HuggingFace!")

### Model Properties

In [ ]:
# Print model properties
print("\n=== Model Properties ===")
print(f"Patch size: {model.patch_size}")
print(f"Max resolution: {model.max_resolution}")
print(f"Preferred resolution: {model.preferred_resolution}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params / 1e6:.1f}M")

## 3. Image Preprocessing

In [ ]:
def preprocess_image_torchhub(
    image: Union[str, Path, Image.Image, np.ndarray],
    model,
    device: str = 'cuda'
) -> torch.Tensor:
    """
    Preprocess image for RADIO model (TorchHub version)
    
    RADIO expects:
    - RGB images
    - Values normalized to [0, 1]
    - Resolution aligned to model's supported resolutions
    """
    # Load image if path provided
    if isinstance(image, (str, Path)):
        image = Image.open(image)
    elif isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    # Convert to RGB
    image = image.convert('RGB')
    
    # Convert to tensor and normalize to [0, 1]
    x = pil_to_tensor(image).to(dtype=torch.float32, device=device)
    x.div_(255.0)
    x = x.unsqueeze(0)  # Add batch dimension: (1, 3, H, W)
    
    # Resize to nearest supported resolution
    nearest_res = model.get_nearest_supported_resolution(*x.shape[-2:])
    x = F.interpolate(x, nearest_res, mode='bilinear', align_corners=False)
    
    return x, nearest_res


def preprocess_image_hf(
    image: Union[str, Path, Image.Image, np.ndarray],
    processor: 'CLIPImageProcessor',
    device: str = 'cuda'
) -> torch.Tensor:
    """
    Preprocess image for RADIO model (HuggingFace version)
    """
    if isinstance(image, (str, Path)):
        image = Image.open(image)
    elif isinstance(image, np.ndarray):
        image = Image.fromarray(image)
    
    image = image.convert('RGB')
    pixel_values = processor(images=image, return_tensors='pt', do_resize=True).pixel_values
    return pixel_values.to(device)

## 4. Feature Extraction

In [ ]:
@torch.no_grad()
def extract_features(
    model,
    image: Union[str, Path, Image.Image, np.ndarray, torch.Tensor],
    device: str = 'cuda',
    feature_format: str = 'NCHW',  # 'NCHW' or 'NLC'
    use_mixed_precision: bool = True
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Extract features from image using RADIO model.
    
    Returns:
        summary: Global image features (B, C)
        spatial_features: Dense features (B, C, H, W) if NCHW, (B, T, D) if NLC
    """
    # Preprocess if not already a tensor
    if not isinstance(image, torch.Tensor):
        x, res = preprocess_image_torchhub(image, model, device)
    else:
        x = image
    
    # Run inference with optional mixed precision
    if use_mixed_precision and device == 'cuda':
        with torch.autocast('cuda', dtype=torch.bfloat16):
            summary, spatial_features = model(x, feature_fmt=feature_format)
    else:
        summary, spatial_features = model(x, feature_fmt=feature_format)
    
    return summary, spatial_features


def get_feature_spatial_dims(model, image_shape: Tuple[int, int]) -> Tuple[int, int]:
    """
    Calculate spatial dimensions of output features.
    """
    H, W = image_shape
    patch_size = model.patch_size
    return H // patch_size, W // patch_size

## 5. Load Your Custom Data

Configure the paths below to point to your data:

In [ ]:
# ============================================
# CONFIGURE YOUR DATA PATHS HERE
# ============================================

# Option 1: Single image and mask
IMAGE_PATH = None  # e.g., "/path/to/your/image.png"
MASK_PATH = None   # e.g., "/path/to/your/mask.png"

# Option 2: Directory with images and masks
IMAGES_DIR = None  # e.g., "/path/to/images/"
MASKS_DIR = None   # e.g., "/path/to/masks/"

# Option 3: Use HuggingFace dataset
HF_DATASET = "nielsr/breast-cancer"  # Default example dataset

# ============================================

In [ ]:
class RadioSegmentationDataset(Dataset):
    """
    Custom dataset for segmentation with RADIO features.
    """
    def __init__(
        self,
        images_dir: Optional[str] = None,
        masks_dir: Optional[str] = None,
        image_paths: Optional[List[str]] = None,
        mask_paths: Optional[List[str]] = None,
        transform=None
    ):
        self.transform = transform
        
        if images_dir and masks_dir:
            images_dir = Path(images_dir)
            masks_dir = Path(masks_dir)
            self.image_paths = sorted(list(images_dir.glob("*.png")) + 
                                       list(images_dir.glob("*.jpg")) +
                                       list(images_dir.glob("*.jpeg")))
            self.mask_paths = [masks_dir / p.name for p in self.image_paths]
        elif image_paths and mask_paths:
            self.image_paths = [Path(p) for p in image_paths]
            self.mask_paths = [Path(p) for p in mask_paths]
        else:
            self.image_paths = []
            self.mask_paths = []
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        mask = Image.open(self.mask_paths[idx]).convert('L')
        
        if self.transform:
            image, mask = self.transform(image, mask)
        
        return {
            'image': image,
            'mask': mask,
            'path': str(self.image_paths[idx])
        }

In [ ]:
# Load data based on configuration
def load_sample_data():
    """Load sample data for testing."""
    
    # Try custom paths first
    if IMAGE_PATH and Path(IMAGE_PATH).exists():
        print(f"Loading single image from: {IMAGE_PATH}")
        image = Image.open(IMAGE_PATH).convert('RGB')
        mask = Image.open(MASK_PATH).convert('L') if MASK_PATH else None
        return [(image, mask, IMAGE_PATH)]
    
    if IMAGES_DIR and Path(IMAGES_DIR).exists():
        print(f"Loading images from directory: {IMAGES_DIR}")
        dataset = RadioSegmentationDataset(IMAGES_DIR, MASKS_DIR)
        return [(d['image'], d['mask'], d['path']) for d in dataset]
    
    # Fall back to HuggingFace dataset
    print(f"Loading HuggingFace dataset: {HF_DATASET}")
    from datasets import load_dataset
    ds = load_dataset(HF_DATASET, split="train")
    
    samples = []
    for i in range(min(5, len(ds))):  # Load first 5 samples
        image = ds[i]['image']
        mask = ds[i].get('label', ds[i].get('mask', None))
        samples.append((image, mask, f"hf_sample_{i}"))
    
    return samples

# Load samples
samples = load_sample_data()
print(f"Loaded {len(samples)} samples")

## 6. Test Feature Extraction

In [ ]:
# Test with first sample
image, mask, path = samples[0]
print(f"Image size: {image.size}")
if mask:
    print(f"Mask size: {mask.size if hasattr(mask, 'size') else np.array(mask).shape}")

# Extract features
x, res = preprocess_image_torchhub(image, model, device)
print(f"\nPreprocessed tensor shape: {x.shape}")
print(f"Target resolution: {res}")

# Get features
summary, spatial_features = extract_features(model, x, device=device, feature_format='NCHW')
print(f"\nSummary shape: {summary.shape}")
print(f"Spatial features shape: {spatial_features.shape}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3 if mask else 2, figsize=(15, 5))

# Original image
axes[0].imshow(image)
axes[0].set_title(f"Original Image\n{image.size}")
axes[0].axis('off')

# Ground truth mask
if mask:
    mask_np = np.array(mask)
    axes[1].imshow(mask_np, cmap='gray')
    axes[1].set_title(f"Ground Truth Mask\n{mask_np.shape}")
    axes[1].axis('off')
    feat_ax = axes[2]
else:
    feat_ax = axes[1]

# Feature map visualization (first 3 channels as RGB)
feat_vis = spatial_features[0, :3].cpu().numpy()
feat_vis = (feat_vis - feat_vis.min()) / (feat_vis.max() - feat_vis.min() + 1e-8)
feat_vis = feat_vis.transpose(1, 2, 0)
feat_ax.imshow(feat_vis)
feat_ax.set_title(f"Feature Map (first 3 channels)\n{spatial_features.shape[-2:]}")
feat_ax.axis('off')

plt.tight_layout()
plt.show()

## 7. Segmentation Approaches with RADIO Features

Several ways to use RADIO features for segmentation:

### 7.1 PCA-based Feature Visualization

In [ ]:
from sklearn.decomposition import PCA

def visualize_features_pca(features: torch.Tensor, n_components: int = 3):
    """
    Visualize high-dimensional features using PCA.
    
    Args:
        features: (B, C, H, W) or (B, T, D) tensor
        n_components: Number of PCA components (3 for RGB visualization)
    """
    # Handle different formats
    if features.ndim == 4:  # NCHW
        B, C, H, W = features.shape
        feat_flat = features[0].permute(1, 2, 0).reshape(-1, C).cpu().numpy()
    else:  # NLC
        B, T, D = features.shape
        H = W = int(np.sqrt(T))
        feat_flat = features[0].cpu().numpy()
        C = D
    
    # Apply PCA
    pca = PCA(n_components=n_components)
    feat_pca = pca.fit_transform(feat_flat)
    
    # Reshape back to spatial
    feat_pca = feat_pca.reshape(H, W, n_components)
    
    # Normalize for visualization
    feat_pca = (feat_pca - feat_pca.min()) / (feat_pca.max() - feat_pca.min() + 1e-8)
    
    return feat_pca, pca.explained_variance_ratio_

# Visualize
feat_pca, var_ratio = visualize_features_pca(spatial_features)
print(f"Explained variance ratio: {var_ratio}")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(image)
plt.title("Original Image")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(feat_pca)
plt.title("PCA Features (RGB)")
plt.axis('off')

plt.tight_layout()
plt.show()

### 7.2 Simple Linear Segmentation Head

In [ ]:
class SimpleSegmentationHead(nn.Module):
    """
    Simple linear segmentation head for RADIO features.
    """
    def __init__(self, in_channels: int, num_classes: int = 2):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, num_classes, kernel_size=1)
    
    def forward(self, features: torch.Tensor, target_size: Tuple[int, int] = None):
        """
        Args:
            features: (B, C, H, W) spatial features from RADIO
            target_size: (H, W) to upsample to original image size
        """
        x = self.conv(features)
        if target_size:
            x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
        return x


# Example: Create a segmentation head
feature_dim = spatial_features.shape[1]  # e.g., 1280 for ViT-H
seg_head = SimpleSegmentationHead(in_channels=feature_dim, num_classes=2).to(device)
print(f"Feature dimension: {feature_dim}")
print(f"Segmentation head created")

In [ ]:
# Quick test (untrained - just for demonstration)
with torch.no_grad():
    pred = seg_head(spatial_features, target_size=image.size[::-1])
    pred_mask = torch.argmax(pred, dim=1).squeeze().cpu().numpy()

print(f"Prediction shape: {pred.shape}")
print(f"Predicted mask shape: {pred_mask.shape}")

### 7.3 K-Means Clustering for Unsupervised Segmentation

In [ ]:
from sklearn.cluster import KMeans

def segment_with_kmeans(
    features: torch.Tensor,
    n_clusters: int = 2,
    target_size: Tuple[int, int] = None
) -> np.ndarray:
    """
    Perform unsupervised segmentation using K-means on RADIO features.
    """
    # Get spatial dimensions
    if features.ndim == 4:  # NCHW
        B, C, H, W = features.shape
        feat_flat = features[0].permute(1, 2, 0).reshape(-1, C).cpu().numpy()
    else:  # NLC
        B, T, D = features.shape
        H = W = int(np.sqrt(T))
        feat_flat = features[0].cpu().numpy()
    
    # K-means clustering
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(feat_flat)
    
    # Reshape to spatial
    seg_mask = labels.reshape(H, W)
    
    # Upsample if needed
    if target_size:
        seg_mask = np.array(Image.fromarray(seg_mask.astype(np.uint8)).resize(
            (target_size[1], target_size[0]), Image.NEAREST
        ))
    
    return seg_mask

# Perform K-means segmentation
kmeans_mask = segment_with_kmeans(spatial_features, n_clusters=2, target_size=image.size[::-1])
print(f"K-means mask shape: {kmeans_mask.shape}")

In [ ]:
# Visualize K-means results
fig, axes = plt.subplots(1, 3 if mask else 2, figsize=(15, 5))

axes[0].imshow(image)
axes[0].set_title("Original Image")
axes[0].axis('off')

if mask:
    axes[1].imshow(np.array(mask), cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[1].axis('off')
    ax_pred = axes[2]
else:
    ax_pred = axes[1]

ax_pred.imshow(kmeans_mask, cmap='viridis')
ax_pred.set_title("K-Means Segmentation")
ax_pred.axis('off')

plt.tight_layout()
plt.show()

### 7.4 Similarity-based Point Prompt Segmentation

In [ ]:
def segment_by_point_similarity(
    features: torch.Tensor,
    point: Tuple[int, int],  # (y, x) in feature map coordinates
    threshold: float = 0.7,
    target_size: Tuple[int, int] = None
) -> np.ndarray:
    """
    Segment regions similar to a point prompt using cosine similarity.
    
    Args:
        features: (B, C, H, W) spatial features
        point: (y, x) coordinates in feature map space
        threshold: Similarity threshold for segmentation
        target_size: (H, W) to upsample result
    """
    B, C, H, W = features.shape
    
    # Get point feature
    y, x = point
    point_feat = features[0, :, y, x]  # (C,)
    
    # Compute cosine similarity to all positions
    feat_flat = features[0].reshape(C, -1)  # (C, H*W)
    point_feat = point_feat.unsqueeze(1)  # (C, 1)
    
    # Normalize and compute similarity
    feat_norm = F.normalize(feat_flat, dim=0)
    point_norm = F.normalize(point_feat, dim=0)
    similarity = torch.mm(point_norm.T, feat_norm).squeeze()  # (H*W,)
    
    # Reshape and threshold
    sim_map = similarity.reshape(H, W).cpu().numpy()
    seg_mask = (sim_map > threshold).astype(np.uint8)
    
    # Upsample if needed
    if target_size:
        sim_map = np.array(Image.fromarray((sim_map * 255).astype(np.uint8)).resize(
            (target_size[1], target_size[0]), Image.BILINEAR
        )) / 255.0
        seg_mask = np.array(Image.fromarray(seg_mask * 255).resize(
            (target_size[1], target_size[0]), Image.NEAREST
        )) // 255
    
    return sim_map, seg_mask

# Example: click on center of the feature map
feat_h, feat_w = spatial_features.shape[-2:]
center_point = (feat_h // 2, feat_w // 2)

sim_map, point_seg = segment_by_point_similarity(
    spatial_features, 
    point=center_point,
    threshold=0.5,
    target_size=image.size[::-1]
)

print(f"Feature map size: {feat_h}x{feat_w}")
print(f"Point location: {center_point}")

In [ ]:
# Visualize point-based segmentation
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image)
# Mark the point on the image
point_y = int(center_point[0] / feat_h * image.size[1])
point_x = int(center_point[1] / feat_w * image.size[0])
axes[0].scatter([point_x], [point_y], c='red', s=100, marker='+')
axes[0].set_title("Original Image with Point")
axes[0].axis('off')

axes[1].imshow(sim_map, cmap='hot')
axes[1].set_title("Similarity Map")
axes[1].axis('off')

axes[2].imshow(point_seg, cmap='gray')
axes[2].set_title("Thresholded Segmentation")
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 8. Training a Simple Segmentation Model

Train a linear segmentation head on RADIO features:

In [ ]:
class RadioSegmentationModel(nn.Module):
    """
    Full segmentation model using frozen RADIO backbone + trainable head.
    """
    def __init__(self, radio_model, num_classes: int = 2, freeze_backbone: bool = True):
        super().__init__()
        self.backbone = radio_model
        self.freeze_backbone = freeze_backbone
        
        # Freeze backbone if specified
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        
        # Determine feature dimension (run a dummy forward pass)
        with torch.no_grad():
            dummy = torch.zeros(1, 3, 512, 512).to(next(self.backbone.parameters()).device)
            _, feat = self.backbone(dummy, feature_fmt='NCHW')
            self.feature_dim = feat.shape[1]
        
        # Segmentation head
        self.seg_head = nn.Sequential(
            nn.Conv2d(self.feature_dim, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, num_classes, kernel_size=1)
        )
    
    def forward(self, x: torch.Tensor, target_size: Tuple[int, int] = None):
        # Extract features
        if self.freeze_backbone:
            with torch.no_grad():
                _, features = self.backbone(x, feature_fmt='NCHW')
        else:
            _, features = self.backbone(x, feature_fmt='NCHW')
        
        # Segmentation head
        logits = self.seg_head(features)
        
        # Upsample to target size
        if target_size:
            logits = F.interpolate(logits, size=target_size, mode='bilinear', align_corners=False)
        
        return logits


# Create the model
seg_model = RadioSegmentationModel(model, num_classes=2, freeze_backbone=True).to(device)
print(f"Feature dimension: {seg_model.feature_dim}")
print(f"Trainable parameters: {sum(p.numel() for p in seg_model.parameters() if p.requires_grad) / 1e6:.2f}M")

In [ ]:
def train_step(
    model: RadioSegmentationModel,
    image: torch.Tensor,
    mask: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    criterion = nn.CrossEntropyLoss()
):
    """
    Single training step.
    """
    model.train()
    optimizer.zero_grad()
    
    # Forward pass
    logits = model(image, target_size=mask.shape[-2:])
    
    # Compute loss
    loss = criterion(logits, mask)
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    return loss.item()


def evaluate(
    model: RadioSegmentationModel,
    image: torch.Tensor,
    mask: torch.Tensor
) -> Tuple[float, np.ndarray]:
    """
    Evaluate model on single sample.
    """
    model.eval()
    with torch.no_grad():
        logits = model(image, target_size=mask.shape[-2:])
        pred = torch.argmax(logits, dim=1)
        
        # Calculate IoU
        intersection = ((pred == 1) & (mask == 1)).sum().float()
        union = ((pred == 1) | (mask == 1)).sum().float()
        iou = (intersection / (union + 1e-8)).item()
        
    return iou, pred.cpu().numpy()

In [ ]:
# Example training loop (on single sample for demonstration)
if mask is not None:
    # Prepare data
    train_image = x  # Already preprocessed
    train_mask = torch.from_numpy(
        np.array(mask.resize((res[1], res[0]), Image.NEAREST))
    ).long().unsqueeze(0).to(device)
    
    # Binarize mask (assuming positive values are foreground)
    train_mask = (train_mask > 0).long()
    
    # Setup optimizer
    optimizer = torch.optim.Adam(seg_model.seg_head.parameters(), lr=1e-3)
    
    # Train for a few iterations
    print("Training segmentation head...")
    losses = []
    for epoch in range(50):
        loss = train_step(seg_model, train_image, train_mask, optimizer)
        losses.append(loss)
        if (epoch + 1) % 10 == 0:
            iou, pred = evaluate(seg_model, train_image, train_mask)
            print(f"Epoch {epoch+1}: Loss = {loss:.4f}, IoU = {iou:.4f}")
    
    print("\nTraining complete!")
else:
    print("No mask available for training demonstration.")

In [ ]:
# Visualize training results
if mask is not None:
    iou, pred = evaluate(seg_model, train_image, train_mask)
    
    # Upsample prediction to original size
    pred_resized = np.array(Image.fromarray(pred[0].astype(np.uint8)).resize(
        image.size, Image.NEAREST
    ))
    
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    axes[0].imshow(image)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    axes[1].imshow(np.array(mask), cmap='gray')
    axes[1].set_title("Ground Truth")
    axes[1].axis('off')
    
    axes[2].imshow(pred_resized, cmap='gray')
    axes[2].set_title(f"Prediction (IoU: {iou:.3f})")
    axes[2].axis('off')
    
    # Training loss curve
    axes[3].plot(losses)
    axes[3].set_xlabel('Epoch')
    axes[3].set_ylabel('Loss')
    axes[3].set_title('Training Loss')
    
    plt.tight_layout()
    plt.show()

## 9. Batch Processing Multiple Images

In [ ]:
@torch.no_grad()
def process_batch(
    model,
    images: List[Image.Image],
    device: str = 'cuda',
    batch_size: int = 4
) -> List[Tuple[torch.Tensor, torch.Tensor]]:
    """
    Process multiple images in batches.
    """
    results = []
    
    for i in range(0, len(images), batch_size):
        batch_images = images[i:i+batch_size]
        
        # Preprocess batch
        batch_tensors = []
        for img in batch_images:
            x, res = preprocess_image_torchhub(img, model, device)
            batch_tensors.append(x)
        
        # Concatenate (assuming same size after preprocessing)
        batch = torch.cat(batch_tensors, dim=0)
        
        # Extract features
        with torch.autocast('cuda', dtype=torch.bfloat16):
            summary, features = model(batch, feature_fmt='NCHW')
        
        # Split results
        for j in range(len(batch_images)):
            results.append((summary[j:j+1], features[j:j+1]))
    
    return results

# Process all samples
if len(samples) > 1:
    images_only = [s[0] for s in samples]
    batch_results = process_batch(model, images_only, device=device)
    print(f"Processed {len(batch_results)} images")
    print(f"Feature shape per image: {batch_results[0][1].shape}")

## 10. Compare Multiple Samples

In [ ]:
# Visualize all samples with their K-means segmentation
n_samples = min(5, len(samples))
fig, axes = plt.subplots(n_samples, 3, figsize=(15, 5*n_samples))
if n_samples == 1:
    axes = axes.reshape(1, -1)

for i, (img, msk, path) in enumerate(samples[:n_samples]):
    # Get features
    x_i, res_i = preprocess_image_torchhub(img, model, device)
    _, feat_i = extract_features(model, x_i, device=device, feature_format='NCHW')
    
    # K-means segmentation
    kmeans_seg = segment_with_kmeans(feat_i, n_clusters=2, target_size=img.size[::-1])
    
    # Plot
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(f"Image {i+1}")
    axes[i, 0].axis('off')
    
    if msk is not None:
        axes[i, 1].imshow(np.array(msk), cmap='gray')
        axes[i, 1].set_title("Ground Truth")
    else:
        axes[i, 1].text(0.5, 0.5, 'No mask', ha='center', va='center')
        axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis('off')
    
    axes[i, 2].imshow(kmeans_seg, cmap='viridis')
    axes[i, 2].set_title("K-Means Segmentation")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

## 11. Save/Load Model Weights

In [ ]:
# Save segmentation head weights (not the full RADIO backbone)
def save_seg_head(model: RadioSegmentationModel, path: str):
    """Save only the segmentation head weights."""
    torch.save(model.seg_head.state_dict(), path)
    print(f"Saved segmentation head to {path}")

def load_seg_head(model: RadioSegmentationModel, path: str):
    """Load segmentation head weights."""
    model.seg_head.load_state_dict(torch.load(path))
    print(f"Loaded segmentation head from {path}")

# Example usage:
# save_seg_head(seg_model, "radio_seg_head.pth")
# load_seg_head(seg_model, "radio_seg_head.pth")

## Summary

This notebook demonstrated how to use NVIDIA's C-RADIOv4 for segmentation:

1. **Model Loading**: Via TorchHub or HuggingFace
2. **Feature Extraction**: Get dense spatial features from images
3. **Segmentation Approaches**:
   - PCA visualization
   - K-means clustering (unsupervised)
   - Point-based similarity segmentation
   - Trainable linear segmentation head
4. **Training**: Fine-tune a simple segmentation head on frozen RADIO features

### Next Steps

- Train on your full dataset with proper train/val splits
- Try different segmentation head architectures (UPerNet, FPN, etc.)
- Experiment with unfreezing some RADIO layers for fine-tuning
- Use RADIO with SAM-style decoder for interactive segmentation

In [ ]:
# Clean up
if device == 'cuda':
    torch.cuda.empty_cache()
    print(f"GPU memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB allocated")